# Cinco mapas de Santa Catarina
NO₂ observado e CO₂ direto e dos insumos encadeados com 67 perfis setoriais estimados para 20 microrregiões. Base monetária 2015, fatores julgamentais elaborados em 2026 com fontes de vários anos. Não é uma MIP regional observada. Ilhas preservadas; água cartografada excluída.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
from IPython.display import SVG, display
pasta = Path.cwd() if Path('satelite.py').exists() else Path.cwd() / 'microrregioes_sc'
sys.path.insert(0, str(pasta.resolve()))
from territorio import carregar_terra
from satelite import adquirir_recorte, agregar_regioes
from mip import ler_p
from regionalizacao import produzir_pesos, ler_pesos, regionalizar
regioes = carregar_terra()

## Satélite → microrregião terrestre
A média usa apenas a área terrestre de interseção: Σ(A × NO₂) / Σ(A), sobre células válidas. Os valores temporais são as composições anuais FMI; não se repondera a média espacial pelo peso HARP.

In [ ]:
recorte = adquirir_recorte(2023)
with np.load(recorte) as grade:
    resultados_no2 = agregar_regioes(regioes, grade['latitude'], grade['longitude'],
                                     grade['no2_pmolec_cm2'], grade['peso_harp'])
resultados_no2

## Mesma produção: próprias + insumos = totais
Para a produção bruta x[j] da atividade j: diretas[j] = gamma[j] × x[j]; totais[j] = (gamma @ L)[j] × x[j]. Os insumos de todas as etapas são (gamma @ (L-I))[j] × x[j]. L = (I-A)^-1 representa a cadeia nacional, sem emissões ocorridas no exterior. A soma das linhas de P confere o componente direto; a soma das colunas de P (demanda final) não é usada como total.

As pegadas se sobrepõem entre setores: o total regional é uma soma de pegadas da produção, não um inventário territorial sem dupla contagem. A diagonal de P não é o componente direto.

In [ ]:
from mip import ler_tecnologia
atividades, P = ler_p()
_, gamma, x, L = ler_tecnologia()
emissoes_diretas = gamma * x
emissoes_insumos = (gamma @ (L - np.eye(len(x)))) * x
emissoes_totais = (gamma @ L) * x
np.testing.assert_allclose(emissoes_diretas, P.sum(axis=1), atol=1e-8)
np.testing.assert_allclose(emissoes_totais, emissoes_diretas + emissoes_insumos)
assert (emissoes_totais >= emissoes_diretas - 1e-8).all()
print('Gg CO₂; diretas e soma de pegadas:', emissoes_diretas.sum(), emissoes_totais.sum())

## VAB observado → hipóteses setoriais → pesos
O fator por atividade e microrregião é estimado, não medido. As fontes orientam vocações; a magnitude de todos os fatores é julgamental. Normalizamos cada atividade para preservar sua participação estadual anterior: w = soma(base) × base × fator / soma(base × fator). A escala SC/Brasil permanece baseada em quatro grupos de VAB, inclusive para atividades raras.
O CSV de fatores tem todos os 1.340 valores, justificativas e fontes. Confiança quantitativa baixa; fontes posteriores a 2015 não são observações daquele ano.

In [ ]:
from regionalizacao import ler_fatores, ajustar_distribuicao
fatores, justificativas = ler_fatores(atividades)
base = ler_pesos(pasta / 'dados/pesos_vab_quatro_grupos_2015.csv', atividades)
# Cada coluna é uma atividade: normaliza regiões e preserva o subtotal SC.
massa = base * fatores
pesos_estimados = massa / massa.sum(axis=0)[None, :] * base.sum(axis=0)[None, :]
np.testing.assert_allclose(pesos_estimados.sum(axis=0), base.sum(axis=0))
print('Fatores estimados:', fatores.shape, 'Exemplo de justificativa:', justificativas['6280'])
arquivo_pesos = pasta / 'dados/pesos_setores_microrregioes_2015.csv'
if not arquivo_pesos.exists():
    produzir_pesos()
pesos = ler_pesos(arquivo_pesos, atividades)
np.testing.assert_allclose(pesos, pesos_estimados)
diretas_setor_regiao = pesos * emissoes_diretas[None, :]
insumos_setor_regiao = pesos * emissoes_insumos[None, :]
totais_setor_regiao = pesos * emissoes_totais[None, :]
np.testing.assert_allclose(totais_setor_regiao, diretas_setor_regiao + insumos_setor_regiao)
exportado = regionalizar()
np.testing.assert_allclose([r['emissoes_diretas'] for r in exportado], diretas_setor_regiao.sum(axis=1))
np.testing.assert_allclose([r['emissoes_totais'] for r in exportado], totais_setor_regiao.sum(axis=1))
print('SC, Gg CO₂:', diretas_setor_regiao.sum(), insumos_setor_regiao.sum(), totais_setor_regiao.sum())

## Figura e entradas
A/B compartilham a escala de NO₂. C/D mostram diretas/insumos na mesma paleta Viridis invertida e escala linear em Gg CO₂, de zero ao maior valor das duas contas: cores iguais representam valores iguais. Não interpretar associação espacial entre anos e poluentes diferentes como validação causal. O inventário abaixo distingue as fontes anteriores à P das entradas diretas do painel.
O painel D usa apenas `emissoes_insumos` (total menos diretas), sem repetir as emissões próprias de C. Todos os mapas usam Viridis invertida; E tem escala própria de VAB.

A escala de NO₂ vai do mínimo ao máximo observado nas células terrestres válidas, com os mesmos limites em A e B. As unidades e médias são preservadas. Cores indicam contraste espacial, não níveis seguros ou um fundo natural.


## VAB publicado pelo IBGE → painel E
Somamos agropecuária, indústria, serviços exceto administração pública e administração pública (SIDRA 5938). Não é PIB: faltam os impostos líquidos sobre produtos. Mil reais / 1.000.000 = bilhões de reais correntes de 2015. O VAB já ancora os pesos de C/D, portanto não oferece validação independente.


In [ ]:
from regionalizacao import carregar_vab
vab = carregar_vab()
componentes = ['agropecuaria', 'industria', 'servicos', 'administracao_publica']
vab_bilhoes = np.array([sum(r[f'vab_{g}_mil_reais'] for g in componentes)/1e6 for r in vab])
np.testing.assert_allclose(vab_bilhoes, [r['vab_total_bilhoes_reais'] for r in vab])
print('VAB das 20 microrregiões, R$ bilhões de 2015:', vab_bilhoes.sum())


In [ ]:
from painel import gerar_painel
gerar_painel(2023)
display(SVG(filename=str(pasta / 'figuras/painel_cinco_mapas_2023_viridis.svg')))
print((pasta / 'dados/entradas.csv').read_text(encoding='utf-8'))